# Concept Bottleneck Models

In the tale of the teenager and the deep learning model, we see a critical gap between how traditional deep learning (DL) models and humans, like teenagers, approach decisions. While both drivers stopped at the intersection, the teenager could explain how she arrived at that conclusion using ***concepts*** like the light color and the presence (or absence) of the ambulance. In contrast, the DL model, despite making the correct decision and having been trained on millions of instances where lights and ambulances were around, lacked a satisfying decision-making process for the driving evaluator, as the model's answers were based on raw pixel activations. This example highlights the need for developing DL models with a transparent decision-making process like the teenager's, especially in high-stakes fields such as medicine (deciding whether to give a certain treatment), finance (deciding whether to approve a loan), and law (checking whether a hiring system is fair).

> ⚠️ **Warning:** The following paragraphs assumes a basic understanding of probability.

From a technical standpoint, the problem can be described as follows: we aim to model a relationship between a set of input variables $x \in \mathcal{X}$ (such as the image of the road) and a set of output variables corresponding to decisions $y \in \mathcal{Y}$ (whether to cross or stop). The DL model in the tale modeled the relationship as $p(y = \text{cross} \mid x = \text{image’s pixels})$, directly mapping raw image data to a decision. The teenager, instead, modeled the same problem using **higher-level variables — referred to as “concepts”, like the light color and the ambulance — leading to a more human-interpretable decision-making process**. Her reasoning could be expressed as $p(y = \text{cross} \mid c_1 = \text{light color}, c_2 = \text{ambulance})$, where the concepts $c \in \mathcal{C}$ provided insight into **how** she made a particular decision.

To emulate the teenager's approach, we could model a probability distribution similar to $p(y \mid x)$ where output variables $y$ are conditioned on human-understandable concepts $c$ instead of raw features $x$:

$$p(y \mid x) \approx p(y, c | x) = p(y \mid c) \cdot p(c \mid x)$$

In this formulation, the DL model processes the input features $x$ (such as pixels from an image) and maps them to a set of interpretable, high-level variables $c$, known as "***concepts***". These concepts are analogous to the reasoning elements identified by the teenager — such as the traffic light color or the presence of an ambulance. The second part of the model then uses these concepts to determine the final outcome $y$ (whether to cross or stop). This class of models is known as a **Concept Bottleneck Model (CBM)** {cite}`koh2020concept`. CBMs parametrize the conditional distributions with a pair of neural networks:
- A "*concept encoder*" $g$  that takes as an input a sample $x$, and predicts concepts $c$. This network parametrizes the concept distribution.
- A "*task predictor*" $f$ that takes as an input a set of concepts $c$ and predicts an output label $y$. This network parametrizes the output distribution.

As a result, a CBM models the concept and output probability distributions as follows:

$$p(y, c \mid x; \theta_g, \theta_f) = p(y \mid c; \theta_f) \cdot p(c \mid x; \theta_g)$$

Given a concept-based dataset of i.i.d. triples (input, concepts, task) $\mathcal{D} = \{(\hat{x}, \hat{c}, \hat{y})\}$, the CBM’s parameters ($\theta_g$ and $\theta_f$) are usually optimized via gradient descent by maximizing the log-likelihood<sup>1</sup>:

$$\max_{\theta_g, \theta_f} \mathcal{L}(\theta_g, \theta_f) = \sum_{(\hat{x}, \hat{c}, \hat{y}) \in \mathcal{D}} \log p(\hat{y}, \hat{c} \mid \hat{x}; \theta_g, \theta_f) =\\
= \sum_{(\hat{x}, \hat{c}, \hat{y}) \in \mathcal{D}} \log p(\hat{y} \mid \hat{c}; \theta_f) + \log p(\hat{c} \mid \hat{x}; \theta_g)$$

The following coding practice introduces you to implementing CBMs and shows how to query CBMs to understand the model's decision-making process.


**ADD JOINT, SEQUENTIAL, INDEPENDENT TRAINING** [Mateo]


## Coding practice

In this practice, we implement a Concept Bottleneck Model (CBM) for a simple traffic light scenario where decisions to cross or stop depend on two concepts: the traffic light being green and the presence of an ambulance. The model predicts these concepts and uses them to make decisions.

> ⚠️ **Warning:** This section assumes a basic understanding of programming machine learning scripts using deep learning frameworks, specifically PyTorch. If you're new to PyTorch, consider starting with [these introductory tutorials](https://pytorch.org/tutorials/beginner/basics/intro.html) to get up to speed.


### Step #1: Install PyC

First, we install the necessary Python packages required to implement CBMs. This includes our library `pytorch-concepts` which runs on top of standard deep learning libraries (`PyTorch` and `torch_geometric`).

In [1]:
%%capture
!pip install torch
!pip install torch_geometric
!pip install -i https://test.pypi.org/simple/ --upgrade pytorch-concepts

### Step #2: Load traffic scenario

Next, we load the dataset for our traffic light scenario. This dataset comprises a collection of images of various road intersections under different conditions. Each image is annotated with the following class labels:
*   **Traffic Light Color**: Indicates the current color of the traffic light (e.g., red, yellow, green).
*   **Presence of an Ambulance**: Specifies whether an ambulance is visible in the scene.
*   **Decision to Cross**: Denotes whether the appropriate action is to cross the intersection or to stop.

In [2]:
from torch_concepts.data import TrafficLights

n_samples = 1000

# Loading dataset
dataset = TrafficLights(n_samples=n_samples)
x_train, c_train, y_train, concept_names, task_names = dataset.x_train, dataset.c_train, dataset.y_train, dataset.concept_names, dataset.task_names

# Example of scenario
print(x_train.shape)
print(c_train.shape, c_train[0], concept_names)
print(y_train.shape, y_train[0], task_names)

torch.Size([1000, 10])
torch.Size([1000, 2]) tensor([0., 1.]) ['traffic light green', 'ambulance crossing']
torch.Size([1000, 1]) tensor([0.]) ['cross']


### Step #3: Define the CBM

We are now ready to construct our first CBM. The model comprises three main components:
*    **Encoder**: Reduces the input features to a lower-dimensional latent space.
*    **Concept Scorer**: Predicts the concepts logits (traffic light color and ambulance presence) from the latent representation.
*    **Task Predictor**: Uses the predicted concepts to determine the final decision on whether to cross.

These components are combined sequentially to form the complete CBM architecture.


In [3]:
import torch
from torch_concepts.nn import ConceptEncoder

latent_dims = 5

# Defining the CBM
# The encoder extracts a low-dimensional representation of the input
encoder = torch.nn.Sequential(
    torch.nn.Linear(x_train.shape[1], latent_dims),
    torch.nn.LeakyReLU()
)

# The concept scorer predicts concept logits {traffic light color, ambulance presence}
c_scorer = ConceptEncoder(
    in_features=latent_dims,
    out_concept_dimensions={1: concept_names}
)

# The task predictor determines the value of the downstream label {cross}
y_predictor = torch.nn.Sequential(
    torch.nn.Linear(c_train.shape[1], latent_dims),
    torch.nn.LeakyReLU(),
    torch.nn.Linear(latent_dims, y_train.shape[1])
)

model = torch.nn.Sequential(encoder, c_scorer, y_predictor)

print(model)

Sequential(
  (0): Sequential(
    (0): Linear(in_features=10, out_features=5, bias=True)
    (1): LeakyReLU(negative_slope=0.01)
  )
  (1): ConceptEncoder(
    (encoder): Linear(in_features=5, out_features=2, bias=True)
  )
  (2): Sequential(
    (0): Linear(in_features=2, out_features=5, bias=True)
    (1): LeakyReLU(negative_slope=0.01)
    (2): Linear(in_features=5, out_features=1, bias=True)
  )
)


### Step #4: (Jointly) Train the CBM

We set the training parameters, including the number of epochs (`n_epochs`) and learning rate (`lr`). Using the Adam optimizer and Binary Cross-Entropy Loss, we train the CBM through a standard PyTorch loop. In each epoch, the model performs a forward pass to predict concepts and the final decision, computes the combined loss, performs backpropagation, and updates the model parameters. The concept regularization weight (`concept_reg`) controls the weight of the concept loss w.r.t. the downstream task loss.

In [4]:
n_epochs = 1000
concept_reg = 0.5
lr = 0.01

# Define optimizer and loss function
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
loss_fn = torch.nn.BCELoss()

# Standard PyTorch learning cycle
model.train()
for epoch in range(n_epochs):
   optimizer.zero_grad()

   # Encode input, then predict concept and downstream tasks activations
   emb = encoder(x_train)
   c_pred = c_scorer(emb).sigmoid()
   y_pred = y_predictor(c_pred).sigmoid()

   # Double loss on concepts and tasks
   loss = loss_fn(c_pred, c_train) + concept_reg * loss_fn(y_pred, y_train)
   loss.backward()
   optimizer.step()

   if epoch % 100 == 0:
       print(f"Epoch {epoch}: Loss {loss.item():.2f}")

Epoch 0: Loss 1.07
Epoch 100: Loss 0.51
Epoch 200: Loss 0.32
Epoch 300: Loss 0.14
Epoch 400: Loss 0.08
Epoch 500: Loss 0.06
Epoch 600: Loss 0.05
Epoch 700: Loss 0.04
Epoch 800: Loss 0.03
Epoch 900: Loss 0.03


### Step #5: Trace task prediction back to concept activations

We examine how the concept activations (green light and ambulance presence) influence the downstream task prediction. By testing the model with a sample input, we observe the resulting decision.


In [5]:
model.eval()
print(f"Task ({task_names}): {y_pred[0]>0.5}")
print(f"Concepts ({concept_names}): {c_pred[0]>0.5}")

Task (['cross']): tensor([False])
Concepts (['traffic light green', 'ambulance crossing']): tensor([False,  True])


The model correctly identifies that the traffic light is green and there is no ambulance, and it decides to cross (task prediction is `True`).

### Step #6: Change concept activations to affect task predictions

Finally, we alter the concept activations to different values and observe how these changes influence the downstream task prediction. This demonstrates how the model's decisions are directly tied to specific concept activations.


In [6]:
# Intervene changing the value of the concept "ambulance" to True
c_intervened = c_pred[0].clone().unsqueeze(0)
c_intervened[0, 1] = 1

# Compute new task prediction
y_intervened = y_predictor(c_intervened).sigmoid()

print(f"Concepts: {c_intervened[0]>0.5}")
print(f"Task: {y_intervened[0]>0.5}")

Concepts: tensor([False,  True])
Task: tensor([False])


The model effectively reacts to the concept intervention<sup>2</sup> predicting that in the presence of a green light and of an ambulance, the car should stop (task prediction is `False`).

## Take home message
In summary, in this chapter we demonstrated how to implement Concept Bottleneck Models (CBMs). These models are inherently explainable as:
*    **Task predictions can be traced back to the activation of human-interpretable concepts**. This enables the model to answer the driving evaluator's question by saying, "I decided to cross as I saw a green light and there was no ambulance".
*    **Altering concept values changes the model's decisions**. This allows the model to respond to the evaluator's question with, "In the same scenario, if there is an ambulance, I would cross".


## Next chapter

The next chapter will discuss key metrics used in concept-based interpretability to quantitatively evaluate CBM performance.

## References

Koh, Pang Wei, et al. "Concept bottleneck models." International conference on machine learning. PMLR, 2020.



--------------------
<sup>1</sup>: This optimization problem is known as a CBM's "joint training". Different types of training will be discussed in Chapter 4.

<sup>2</sup>: The ability of CBMs in responding to concept interventions can be used to improve the model's performance by making human experts fix mispredicted concepts. This topic will be discussed in detail in Chapter 3.